# Week 10 — Monday: Linear Regression

**DATA 202 · Calvin University**

Every house in Ames, Iowa has a value on file at the city assessor's office: the number its property tax is based on. The office's records include **2,925 homes sold** from 2006 to 2010, with their size, age, quality and sale price.

> 🎯 **How much is a house worth — and can a straight line tell us?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · Houses for sale |
| ~10 min | 2 · One line (SLO 10A) |
| ~10 min | 3 · How wrong? MAE, RMSE, R² (SLO 10C) |
| ~10 min | 4 · More columns (SLO 10A) |
| ~5 min | 5 · Who does the line get wrong? |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: De Cock (2011), "Ames, Iowa: Alternative to the Boston Housing Data Set," *Journal of Statistics Education* 19(3); from the Ames City Assessor's Office</sub>

---
## 1 · Houses for Sale · ~5 min

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression

ames = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/ames.csv")
X = ames.drop(columns="price")
y = ames["price"]                                        # the label: sale price, in dollars
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)
print(len(X_train), "training houses |", len(X_test), "test houses")
ames.head()

In [ ]:
fig = px.scatter(x=X_train["living_area"], y=y_train, opacity=0.4,
                 labels={"x": "living area (sq ft)", "y": "sale price ($)"},
                 title="2,193 Training Houses: Price vs. Living Area")
fig.show()

> 💬 **Question:** Weeks 6–9 predicted a **class** (survived or not, spam or not). What changes when the label is a **number**?

<details><summary>Answer</summary>

* This is **regression**: the prediction is a number, not a class
* A prediction is rarely exactly right. The question becomes **how far off** it is, not right or wrong
* So accuracy, precision and recall don't apply. We need new metrics (Part 3)
</details>

---
## 2 · One Line (SLO 10A) · ~10 min

> 📄 **Handout 1 — Draw the line.** With a ruler, draw the straight line that best follows the houses. Use it to price a 2,000 sq ft house. 🗣️ Compare with a neighbor: same line?

The computer plays the same game. It tries lines until the misses are as small as possible:

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/10/images/line_fit.gif" width="520" alt="Animation: a red line starts flat and turns step by step until it runs through the middle of a cloud of training and test points; the mean squared error shown below it shrinks.">

<sub>Animation: gbhat.com</sub>

In [ ]:
line = LinearRegression().fit(X_train[["living_area"]], y_train)
print(f"price = {line.intercept_:,.0f} + {line.coef_[0]:,.2f} × living area")
print(f"a 2,000 sq ft house: ${line.predict(pd.DataFrame({'living_area': [2000]}))[0]:,.0f}")

<details><summary>Answer</summary>

* **price = $9,194 + $114.97 × living area**
* **Slope: $115 per square foot.** Each extra square foot goes with about $115 more, on average
* **Intercept: $9,194.** It's the price of a house with 0 sq ft. That makes no sense as a house; it's just where the line crosses the axis
* A 2,000 sq ft house: about **$239,000**. How close was your line?
</details>

**Which line is "best"?** For each house, the **residual** = actual price − predicted price: the vertical miss. **Least squares** picks the line with the smallest **sum of squared residuals**.

$$\text{price} = b_0 + b_1 \times \text{living area} \qquad \text{choose } b_0, b_1 \text{ to minimize } \sum (\text{actual} - \text{predicted})^2$$

---
## 3 · How Wrong? MAE, RMSE, R² (SLO 10C) · ~10 min

> 📄 **Handout 2 — Five test houses.** For each house, compute the residual. Then the **MAE** (mean of the absolute residuals) and the **RMSE** (square root of the mean of the squared residuals).

In [ ]:
five = X_test.loc[[1117, 192, 45, 674, 1770], ["neighborhood", "living_area"]]
five["actual ($1000s)"] = (y_test.loc[five.index] / 1000).round()
five["predicted ($1000s)"] = (line.predict(five[["living_area"]]) / 1000).round()
five["residual"] = five["actual ($1000s)"] - five["predicted ($1000s)"]
five

<details><summary>Answer</summary>

* Residuals: 34, −37, 59, −26, 1 (thousands of dollars)
* **MAE** = (34 + 37 + 59 + 26 + 1) ÷ 5 = **31.4** → typically off by about $31,000
* **RMSE** = √((34² + 37² + 59² + 26² + 1²) ÷ 5) = √1,336.6 = **36.6** → bigger, because squaring makes the $59k miss count extra
</details>

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

predicted = line.predict(X_test[["living_area"]])
average = np.full(len(y_test), y_train.mean())          # a lazy model: always guess the average price
for name, guess in [("line", predicted), ("always the average", average)]:
    print(f"{name:18} | MAE ${mean_absolute_error(y_test, guess):,.0f} | RMSE ${root_mean_squared_error(y_test, guess):,.0f}"
          f" | R² {r2_score(y_test, guess):.2f}")

| metric | means | units |
|---|---|---|
| **MAE** | the typical miss | dollars |
| **RMSE** | like MAE, but big misses count more | dollars |
| **R²** | 1 − (the model's squared misses ÷ the lazy model's squared misses) | none: 1 = perfect, 0 = no better than the average |

<details><summary>Answer</summary>

* The line: MAE **$38,028**, RMSE **$55,055**, R² **0.53**
* Always guessing the average: MAE $59,070, RMSE $80,440, R² ≈ 0
* R² = 0.53: the line removes about half of the lazy model's squared error
</details>

> 📌 **One split:** these are 732 test houses.

---
## 4 · More Columns (SLO 10A) · ~10 min

With more columns, each one gets its own slope: price = b₀ + b₁ × area + b₂ × bedrooms + …

> 🎯 **Predict First:** One extra bedroom: does it add to the price, or take away?

In [ ]:
alone = LinearRegression().fit(X_train[["bedrooms"]], y_train)
both = LinearRegression().fit(X_train[["living_area", "bedrooms"]], y_train)
print(f"bedrooms alone      | ${alone.coef_[0]:>+9,.0f} per bedroom")
print(f"living area too     | ${both.coef_[1]:>+9,.0f} per bedroom | ${both.coef_[0]:,.0f} per sq ft")

> 📄 **Handout 3 — Same size, one more bedroom.** Explain how the same column can be worth +$14,077 in one model and −$31,152 in the other.

<details><summary>Answer</summary>

* **Alone:** houses with more bedrooms are usually bigger, so bedrooms get credit for the size: **+$14,077**
* **With living area:** each slope means "**holding the other columns fixed**." Same square footage, one more bedroom → every room is smaller: **−$31,152**
* A coefficient answers a question about the *other* columns in the model. Change them, and the answer can change sign
</details>

Now all of them: 19 numeric columns plus the neighborhood, in a Week 9 pipeline. `masonry_area` has 23 blanks, and the neighborhood is a word.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric = [c for c in X.columns if c not in ("neighborhood", "year_sold")]
model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["neighborhood"]),
    ])),
    ("line", LinearRegression()),
])
model.fit(X_train, y_train)
predicted = model.predict(X_test)
print(f"test | MAE ${mean_absolute_error(y_test, predicted):,.0f} | RMSE ${root_mean_squared_error(y_test, predicted):,.0f}"
      f" | R² {r2_score(y_test, predicted):.3f}")
print(f"R² on the training houses: {model.score(X_train, y_train):.3f} | 5-fold CV: {cross_val_score(model, X_train, y_train, cv=5).mean():.3f}")

<details><summary>Answer</summary>

* 47 columns: MAE **$18,960**, RMSE **$27,193**, R² **0.885**. The typical miss is half of the one-line model's
* Training 0.878, cross-validation 0.870: almost no gap, so **no overfitting** yet. Wednesday we give the model more freedom
</details>

---
## 5 · Who Does the Line Get Wrong? · ~5 min

The assessor's question isn't only "how far off?" but "off **for whom**?" Percentage error for the test houses, from the cheapest fifth to the priciest fifth:

In [ ]:
errors = pd.DataFrame({
    "price group": pd.qcut(y_test, 5, labels=["cheapest", "2nd", "3rd", "4th", "priciest"]),
    "one line (area)": (line.predict(X_test[["living_area"]]) - y_test) / y_test * 100,
    "all 47 columns": (model.predict(X_test) - y_test) / y_test * 100,
})
errors.groupby("price group", observed=True).mean().round(1)

> 📄 **Handout 4 — Over or under?** Positive = the model says the house is worth **more** than it sold for. Circle the groups the one-line model overvalues and undervalues.

<details><summary>Answer</summary>

* **One line:** the cheapest fifth is valued **50% too high** on average; the priciest fifth, **15% too low**
* **All 47 columns:** +4% and −3%. Much better, but the sign pattern is the same
* If a value like this set property taxes, the owners of the cheapest homes would pay tax on value they don't have, and the owners of the priciest homes on less than they have. Studies of U.S. property assessments have found this same pattern
</details>

> 💬 **Question:** Both models were fit to minimize squared error over all houses. Why might "smallest error on average" still treat cheap and expensive homes differently?

<details><summary>Answer</summary>

* A $50,000 miss is 10% of a $500,000 house and 50% of a $100,000 house. Least squares counts dollars, not percentages, so the expensive houses dominate the fit
* "Best on average" is a choice of **loss**. Another loss (percentage error, or a model per price range) gives another line, and another distribution of who gets it wrong
</details>

---
### Wrap-up

1. **Regression** predicts a number; **least squares** picks the line with the smallest sum of squared residuals
2. A **coefficient** is a slope: dollars per unit, **holding the other columns fixed**
3. **MAE** = the typical miss; **RMSE** = big misses count more; **R²** = share of the lazy model's squared error removed 📌
4. Ask *who* the errors land on, not just how big they are

**Wednesday:** more flexibility, overfitting, and Ridge and Lasso.